# ETL Patterns

## Overview
ETL (Extract, Transform, Load) is the process that moves data from source systems into the data warehouse on a schedule. Reliable ETL is idempotent, auditable, and transactional.

**ETL pipeline stages:**
```
Source DB → [Extract] → Staging table → [Validate] → [Transform] → Fact/Dim tables → [Audit log]
```

**Key principles:**
- Always stage raw data before transforming (preserves audit trail)
- Mark staging rows `pending` → `loaded` to prevent double-loading
- Wrap each batch in a transaction (all-or-nothing)
- Log row counts and rejections in an audit table after every run

---

In [1]:
import sqlite3, json, random
from datetime import date, timedelta

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.executescript("""
-- ── Dimension tables ──────────────────────────────────────────────
CREATE TABLE dim_date (
    date_key    INTEGER PRIMARY KEY,  -- YYYYMMDD
    full_date   TEXT NOT NULL,
    year        INTEGER, quarter INTEGER, month INTEGER,
    month_name  TEXT,    week_of_year INTEGER, day_of_week INTEGER,
    day_name    TEXT,    is_weekend INTEGER, is_holiday INTEGER
);

CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id  TEXT NOT NULL,
    full_name    TEXT, province TEXT, city TEXT,
    segment      TEXT,  -- 'retail','commercial','group'
    risk_tier    TEXT,  -- 'low','medium','high'
    effective_from TEXT, effective_to TEXT, is_current INTEGER DEFAULT 1
);

CREATE TABLE dim_product (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    product_line TEXT,  -- 'auto','home','life','health'
    category     TEXT,
    premium_band TEXT   -- 'basic','standard','premium'
);

CREATE TABLE dim_agent (
    agent_key    INTEGER PRIMARY KEY AUTOINCREMENT,
    agent_id     TEXT NOT NULL,
    agent_name   TEXT,
    region       TEXT,
    channel      TEXT   -- 'direct','broker','online'
);

-- ── Fact table ────────────────────────────────────────────────────
CREATE TABLE fact_policy (
    policy_key      INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        INTEGER REFERENCES dim_date(date_key),
    customer_key    INTEGER REFERENCES dim_customer(customer_key),
    product_key     INTEGER REFERENCES dim_product(product_key),
    agent_key       INTEGER REFERENCES dim_agent(agent_key),
    -- Measures
    premium_amount  REAL,
    coverage_amount REAL,
    claim_amount    REAL DEFAULT 0,
    n_claims        INTEGER DEFAULT 0,
    policy_count    INTEGER DEFAULT 1
);

-- ── Staging table (ETL landing zone) ─────────────────────────────
CREATE TABLE stg_policy_load (
    src_policy_id   TEXT,
    customer_id     TEXT,
    product_id      TEXT,
    agent_id        TEXT,
    policy_date     TEXT,
    premium         REAL,
    coverage        REAL,
    load_ts         TEXT DEFAULT (datetime('now')),
    load_status     TEXT DEFAULT 'pending'
);
""")

# Populate dim_date (2022-2024)
dates = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    dk = int(d.strftime('%Y%m%d'))
    mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    dnames = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    dates.append((dk, str(d), d.year, (d.month-1)//3+1, d.month,
                  mnames[d.month-1], d.isocalendar()[1], d.weekday(),
                  dnames[d.weekday()], 1 if d.weekday()>=5 else 0, 0))
    d += timedelta(days=1)
conn.executemany("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?)", dates)

# Populate dimensions
customers = [
    (1,'C001','Alice Nguyen','ON','Toronto','retail','low','2022-01-01',None,1),
    (2,'C002','Bob Harrington','BC','Vancouver','commercial','medium','2022-01-01',None,1),
    (3,'C003','Fatima Al-Rashid','QC','Montreal','group','low','2022-01-01',None,1),
    (4,'C004','James MacLeod','NS','Halifax','retail','high','2022-01-01',None,1),
    (5,'C005','Mei-Ling Chen','AB','Calgary','commercial','medium','2022-01-01',None,1),
    (6,'C006','David Park','ON','Ottawa','retail','low','2023-06-01',None,1),
]
conn.executemany("INSERT INTO dim_customer VALUES (?,?,?,?,?,?,?,?,?,?)", customers)

products = [
    (1,'P01','Auto Basic','auto','liability','basic'),
    (2,'P02','Auto Comprehensive','auto','comprehensive','standard'),
    (3,'P03','Home Standard','home','dwelling','standard'),
    (4,'P04','Home Premium','home','dwelling','premium'),
    (5,'P05','Term Life 10','life','term','basic'),
    (6,'P06','Whole Life','life','permanent','premium'),
    (7,'P07','Group Health','health','group','standard'),
]
conn.executemany("INSERT INTO dim_product VALUES (?,?,?,?,?,?)", products)

agents = [
    (1,'A01','Sandra Lee','East','direct'),
    (2,'A02','Tom Okafor','West','broker'),
    (3,'A03','Priya Sharma','Central','online'),
    (4,'A04','Marc Tremblay','East','broker'),
]
conn.executemany("INSERT INTO dim_agent VALUES (?,?,?,?,?)", agents)

# Generate fact_policy rows (2022-2024)
random.seed(42)
facts = []
policy_dates = [
    (20220115,20220315,20220601,20220901,20221201),
    (20230115,20230401,20230715,20231001,20231215),
    (20240201,20240401,20240701,20241001,20241201),
]
combos = [
    (20220115,1,1,1,1200,50000,0,0),
    (20220315,1,3,2,1800,180000,0,0),
    (20220601,2,2,1,2400,120000,500,1),
    (20220901,3,5,3,600,200000,0,0),
    (20221201,4,1,4,1400,60000,2200,1),
    (20230115,1,2,2,2600,130000,0,0),
    (20230401,2,4,1,3200,250000,0,0),
    (20230715,5,7,3,4800,0,0,0),
    (20231001,6,3,2,2100,195000,1500,1),
    (20231215,3,6,4,5400,500000,0,0),
    (20240201,4,2,1,1600,70000,800,1),
    (20240401,1,5,3,700,220000,0,0),
    (20240701,5,4,2,3500,280000,0,0),
    (20241001,2,1,1,1100,55000,0,0),
    (20241201,6,7,3,5200,0,0,0),
    (20220115,3,3,2,2000,200000,600,1),
    (20230601,4,4,1,3800,300000,0,0),
    (20240301,5,2,4,2200,115000,0,0),
]
for i, (dk,ck,pk,ak,prem,cov,claim,nclaim) in enumerate(combos):
    facts.append((dk,ck,pk,ak,prem,cov,claim,nclaim,1))
conn.executemany(
    "INSERT INTO fact_policy (date_key,customer_key,product_key,agent_key,premium_amount,coverage_amount,claim_amount,n_claims,policy_count) VALUES (?,?,?,?,?,?,?,?,?)",
    facts)
conn.commit()

print("Data warehouse dataset loaded:")
for t in ['dim_date','dim_customer','dim_product','dim_agent','fact_policy']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22s}: {n} rows")

# Quick sanity check
total = conn.execute("SELECT SUM(premium_amount) FROM fact_policy").fetchone()[0]
print(f"\n  Total premiums: ${total:,.0f}")


Data warehouse dataset loaded:
  dim_date              : 1096 rows
  dim_customer          : 6 rows
  dim_product           : 7 rows
  dim_agent             : 4 rows
  fact_policy           : 18 rows

  Total premiums: $45,600


---
## ETL flow and audit logging

In [2]:
print("=== ETL patterns: Extract, Transform, Load ===")
print()

print("Standard ETL flow for a DWH:")
flow = [
    ("1. Extract",   "Pull data from source systems (transactional DB, files, APIs)"),
    ("2. Stage",     "Load raw data into staging tables (no transformations yet)"),
    ("3. Validate",  "Check nulls, types, referential integrity, duplicates"),
    ("4. Transform", "Apply business rules, lookups, SCD logic, surrogate keys"),
    ("5. Load",      "INSERT/UPSERT into fact and dimension tables"),
    ("6. Audit",     "Record row counts, load timestamps, error counts in audit log"),
    ("7. Swap",      "Swap staging data into production (optional: after validation)"),
]
for step, desc in flow:
    print(f"  {step:<14s}  {desc}")

print()
# Staging pattern demo
print("Staging table audit pattern:")
conn.execute("""
    CREATE TABLE IF NOT EXISTS etl_audit_log (
        run_id       INTEGER PRIMARY KEY AUTOINCREMENT,
        table_name   TEXT,
        run_ts       TEXT DEFAULT (datetime('now')),
        rows_extracted INTEGER,
        rows_loaded    INTEGER,
        rows_rejected  INTEGER,
        status         TEXT,
        notes          TEXT
    )
""")

# Simulate a load run
stg_count = conn.execute("SELECT COUNT(*) FROM stg_policy_load").fetchone()[0]
fact_before = conn.execute("SELECT COUNT(*) FROM fact_policy").fetchone()[0]
conn.execute("""
    INSERT INTO etl_audit_log (table_name, rows_extracted, rows_loaded, rows_rejected, status, notes)
    VALUES ('fact_policy', ?, ?, 1, 'success', '1 row rejected: unknown customer C999')
""", (stg_count, stg_count - 1))
conn.commit()

log = conn.execute("SELECT * FROM etl_audit_log ORDER BY run_id DESC LIMIT 1").fetchone()
print(f"  ETL run: table={log['table_name']}  extracted={log['rows_extracted']}  "
      f"loaded={log['rows_loaded']}  rejected={log['rows_rejected']}  status={log['status']}")
print(f"  Notes: {log['notes']}")


=== ETL patterns: Extract, Transform, Load ===

Standard ETL flow for a DWH:
  1. Extract      Pull data from source systems (transactional DB, files, APIs)
  2. Stage        Load raw data into staging tables (no transformations yet)
  3. Validate     Check nulls, types, referential integrity, duplicates
  4. Transform    Apply business rules, lookups, SCD logic, surrogate keys
  5. Load         INSERT/UPSERT into fact and dimension tables
  6. Audit        Record row counts, load timestamps, error counts in audit log
  7. Swap         Swap staging data into production (optional: after validation)

Staging table audit pattern:
  ETL run: table=fact_policy  extracted=0  loaded=-1  rejected=1  status=success
  Notes: 1 row rejected: unknown customer C999


---
## Upsert and incremental load

In [3]:
print("=== Upsert: INSERT OR UPDATE for dimension maintenance ===")
print()

# SQLite INSERT OR REPLACE (upsert)
print("SQLite: INSERT OR REPLACE (SCD1 upsert):")
conn.execute("""
    INSERT OR REPLACE INTO dim_product (product_key, product_id, product_name, product_line, category, premium_band)
    VALUES (2, 'P02', 'Auto Comprehensive Plus', 'auto', 'comprehensive', 'premium')
""")
conn.commit()
updated = conn.execute("SELECT product_name, premium_band FROM dim_product WHERE product_id='P02'").fetchone()
print(f"  P02 after upsert: name={updated['product_name']}  band={updated['premium_band']}")

print()
print("PostgreSQL: ON CONFLICT DO UPDATE (preferred upsert syntax):")
print("""
  -- Upsert dimension member:
  INSERT INTO dim_product (product_id, product_name, product_line, category, premium_band)
  VALUES ('P02', 'Auto Comprehensive Plus', 'auto', 'comprehensive', 'premium')
  ON CONFLICT (product_id)
  DO UPDATE SET
      product_name = EXCLUDED.product_name,
      premium_band = EXCLUDED.premium_band,
      updated_at   = NOW()
  WHERE  dim_product.product_name IS DISTINCT FROM EXCLUDED.product_name
      OR dim_product.premium_band  IS DISTINCT FROM EXCLUDED.premium_band;
  -- IS DISTINCT FROM: only update if something actually changed
  -- Avoids unnecessary writes and triggers
""")

print()
print("=== Incremental load: only new records ===")
print("""
  -- Load only rows newer than last successful run:
  SELECT MAX(run_ts) AS last_run FROM etl_audit_log
  WHERE  table_name = 'fact_policy' AND status = 'success';

  -- Extract from source using high-watermark:
  INSERT INTO stg_policy_load (...)
  SELECT *
  FROM   source_policies
  WHERE  created_at > (SELECT MAX(run_ts) FROM etl_audit_log
                       WHERE table_name='fact_policy' AND status='success');

  -- PostgreSQL: track max date_key loaded
  INSERT INTO fact_policy (...)
  SELECT ...
  FROM   stg_policy_load s
  WHERE  CAST(REPLACE(s.policy_date,'-','') AS INT)
         > (SELECT COALESCE(MAX(date_key),0) FROM fact_policy);
""")


=== Upsert: INSERT OR UPDATE for dimension maintenance ===

SQLite: INSERT OR REPLACE (SCD1 upsert):
  P02 after upsert: name=Auto Comprehensive Plus  band=premium

PostgreSQL: ON CONFLICT DO UPDATE (preferred upsert syntax):

  -- Upsert dimension member:
  INSERT INTO dim_product (product_id, product_name, product_line, category, premium_band)
  VALUES ('P02', 'Auto Comprehensive Plus', 'auto', 'comprehensive', 'premium')
  ON CONFLICT (product_id)
  DO UPDATE SET
      product_name = EXCLUDED.product_name,
      premium_band = EXCLUDED.premium_band,
      updated_at   = NOW()
  WHERE  dim_product.product_name IS DISTINCT FROM EXCLUDED.product_name
      OR dim_product.premium_band  IS DISTINCT FROM EXCLUDED.premium_band;
  -- IS DISTINCT FROM: only update if something actually changed
  -- Avoids unnecessary writes and triggers


=== Incremental load: only new records ===

  -- Load only rows newer than last successful run:
  SELECT MAX(run_ts) AS last_run FROM etl_audit_log
  WHERE

---
## Truncate-reload vs incremental

In [4]:
print("=== Truncate-reload vs incremental load ===")
print()

strategies = [
    ("Truncate-reload",
     "DELETE all rows from fact table; reload from scratch",
     ["Small fact tables (< 1M rows)", "Source data can be fully re-extracted",
      "Data corrections require full re-run", "Simple, no watermark logic needed"],
     ["Loses data if source system doesn't retain history",
      "Slow for large tables", "Downtime during reload window"]),
    ("Incremental (append)",
     "Add only new rows using high-watermark or change date",
     ["Large fact tables (millions+ rows)", "Append-only events (policies issued)",
      "Source retains last N days data only"],
     ["Requires reliable watermark column", "Misses late-arriving data",
      "More complex error handling"]),
    ("Incremental (upsert)",
     "INSERT new rows; UPDATE changed rows; DELETE removed rows",
     ["Dimensions that change (SCD1)", "Source data can be corrected/deleted",
      "Near-real-time requirements"],
     ["Most complex", "Requires CDC or full outer join to detect deletes",
      "Harder to audit"]),
]
for name, desc, when_to_use, challenges in strategies:
    print(f"  {name}:")
    print(f"    {desc}")
    print(f"    Use when:   {when_to_use[0]}")
    for w in when_to_use[1:]:
        print(f"                {w}")
    print(f"    Challenges: {challenges[0]}")
    for c in challenges[1:]:
        print(f"                {c}")
    print()

print("PostgreSQL truncate-reload pattern:")
print("""
  BEGIN;
    TRUNCATE fact_policy;
    INSERT INTO fact_policy (date_key, customer_key, ...)
    SELECT ...
    FROM   stg_policy_load s
    JOIN   dim_customer c ON ...
    JOIN   dim_product  p ON ...
    JOIN   dim_date     d ON ...;
    -- If successful, COMMIT; if not, ROLLBACK and nothing changes
  COMMIT;
""")


=== Truncate-reload vs incremental load ===

  Truncate-reload:
    DELETE all rows from fact table; reload from scratch
    Use when:   Small fact tables (< 1M rows)
                Source data can be fully re-extracted
                Data corrections require full re-run
                Simple, no watermark logic needed
    Challenges: Loses data if source system doesn't retain history
                Slow for large tables
                Downtime during reload window

  Incremental (append):
    Add only new rows using high-watermark or change date
    Use when:   Large fact tables (millions+ rows)
                Append-only events (policies issued)
                Source retains last N days data only
    Challenges: Requires reliable watermark column
                Misses late-arriving data
                More complex error handling

  Incremental (upsert):
    INSERT new rows; UPDATE changed rows; DELETE removed rows
    Use when:   Dimensions that change (SCD1)
               

---
## Common Pitfalls

**1. Transforming in the source extract step**
Applying business rules during the extract (before staging) means you cannot audit or re-process the raw data if a rule was wrong. Always stage raw data first, transform in a separate step, and retain the raw staging data for debugging.

**2. No idempotency: re-running ETL creates duplicates**
An ETL job that can be safely re-run without side effects is idempotent. Use `INSERT OR IGNORE`, `ON CONFLICT DO NOTHING`, or `load_status` flags to ensure re-runs don't insert duplicate fact rows.

**3. High-watermark drift due to late-arriving data**
Using `MAX(created_at)` as the watermark misses rows with `created_at` backdated (late-arriving data, system clock skew). Use a lookback window: `WHERE created_at >= last_run_ts - INTERVAL '2 hours'` and deduplicate with ON CONFLICT.

**4. Not logging rejected rows**
Silently skipping rows that fail dimension lookups (unknown customer IDs) hides data quality problems. Log all rejected rows to a `stg_rejected` table with a reason code. Review rejection rates after every load.

**5. ETL job not transactional**
If an ETL job loads 500K rows and fails on row 400K, without a transaction the first 400K are committed and the remaining 100K are missing. Wrap each ETL batch in a transaction: BEGIN → load → COMMIT (or ROLLBACK on error).

**6. Not vacuuming/analysing after large loads (PostgreSQL)**
After loading millions of rows, PostgreSQL table statistics are stale. Queries may use inefficient plans until `ANALYZE fact_policy` is run. Autovacuum usually handles this, but for large batch loads, run `ANALYZE` explicitly after the load completes.


---
*sql_methods_library - Samantha McGarrigle*